# Laboratory Task 6 — CNN Implementation
Name: Emmanuel R. Sevilla  
DS4A

**Instruction:** Convert the given CNN architecture diagram into a PyTorch CNN.

The code follows the diagram, including its kernel sizes, strides, padding, dropout, and fully connected layers.

## Architecture from the diagram

Input: **1 × 28 × 28** (batch size 32)

| Layer | Settings |
|---|---|
| Conv1 + ReLU | 1 → 32 channels, kernel 3×3, stride 1, padding 1 |
| MaxPool1 | kernel 2×2, stride 1, padding 1 |
| Conv2 + ReLU | 32 → 64 channels, kernel 3×3, stride 1, padding 1 |
| Conv3 + ReLU | 64 → 128 channels, kernel 3×3, stride 1, padding 1 |
| Conv4 + ReLU | 128 → 256 channels, kernel 3×3, stride 1, padding 1 |
| MaxPool2 | kernel 2×2, stride 1, padding 0 |
| Dropout | p = 0.2 |
| Flatten | — |
| FCN1 + ReLU | flattened features → 1000 |
| FCN2 + ReLU | 1000 → 500 |
| FCN3 + Softmax | 500 → 10 |

There are 10 output classes, so FCN3 has 10 outputs.

## 1. Imports

In [1]:
import torch
import torch.nn as nn

## 2. Work out the feature-map sizes

Output size = floor((Input + 2P − K) / S) + 1, where P is padding, K is kernel size, and S is stride.

In [2]:
def out_size(size, k, s, p):
    return (size + 2 * p - k) // s + 1

s = 28
s = out_size(s, 3, 1, 1); print("After Conv1   :", s)
s = out_size(s, 2, 1, 1); print("After MaxPool1:", s)
s = out_size(s, 3, 1, 1); print("After Conv2   :", s)
s = out_size(s, 3, 1, 1); print("After Conv3   :", s)
s = out_size(s, 3, 1, 1); print("After Conv4   :", s)
s = out_size(s, 2, 1, 0); print("After MaxPool2:", s)

flat = 256 * s * s
print("Flatten size  :", flat)

After Conv1   : 28
After MaxPool1: 29
After Conv2   : 29
After Conv3   : 29
After Conv4   : 29
After MaxPool2: 28
Flatten size  : 200704


## 3. Build the CNN

In [3]:
class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, 3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, stride=1, padding=1),

            nn.Conv2d(32, 64, 3, stride=1, padding=1),
            nn.ReLU(),
            nn.Conv2d(64, 128, 3, stride=1, padding=1),
            nn.ReLU(),
            nn.Conv2d(128, 256, 3, stride=1, padding=1),
            nn.ReLU(),

            nn.MaxPool2d(2, stride=1, padding=0),
            nn.Dropout(p=0.2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256 * 28 * 28, 1000),
            nn.ReLU(),
            nn.Linear(1000, 500),
            nn.ReLU(),
            nn.Linear(500, 10),
            nn.Softmax(dim=1),
        )

    def forward(self, x):
        return self.classifier(self.features(x))

## 4. Create the model

In [4]:
model = CNN()
print(model)

CNN(
  (features): Sequential(
    (0): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=1, padding=1, dilation=1, ceil_mode=False)
    (3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU()
    (5): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (6): ReLU()
    (7): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (8): ReLU()
    (9): MaxPool2d(kernel_size=2, stride=1, padding=0, dilation=1, ceil_mode=False)
    (10): Dropout(p=0.2, inplace=False)
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=200704, out_features=1000, bias=True)
    (2): ReLU()
    (3): Linear(in_features=1000, out_features=500, bias=True)
    (4): ReLU()
    (5): Linear(in_features=500, out_features=10, bias=True)
    (6): Softmax(dim=1)
  )
)


## 5. Test with a dummy batch

The diagram uses a batch of 32, so a random input of shape `(32, 1, 28, 28)` is used.

In [5]:
dummy = torch.randn(32, 1, 28, 28)

model.eval()
with torch.no_grad():
    out = model(dummy)

print("Input shape :", dummy.shape)
print("Output shape:", out.shape)
print("Each row sums to 1:", torch.allclose(out.sum(dim=1), torch.ones(32), atol=1e-4))

Input shape : torch.Size([32, 1, 28, 28])
Output shape: torch.Size([32, 10])
Each row sums to 1: True


## 6. Shape after each layer

In [6]:
x = dummy
with torch.no_grad():
    for layer in list(model.features) + list(model.classifier):
        x = layer(x)
        if not isinstance(layer, (nn.ReLU, nn.Softmax)):
            print(f"{layer.__class__.__name__:10s} -> {tuple(x.shape)}")

Conv2d     -> (32, 32, 28, 28)
MaxPool2d  -> (32, 32, 29, 29)
Conv2d     -> (32, 64, 29, 29)
Conv2d     -> (32, 128, 29, 29)
Conv2d     -> (32, 256, 29, 29)
MaxPool2d  -> (32, 256, 28, 28)
Dropout    -> (32, 256, 28, 28)
Flatten    -> (32, 200704)
Linear     -> (32, 1000)
Linear     -> (32, 500)
Linear     -> (32, 10)


## Final shapes (batch size 32)

| Layer | Output shape |
|---|---|
| Input | `(32, 1, 28, 28)` |
| Conv1 | `(32, 32, 28, 28)` |
| MaxPool1 | `(32, 32, 29, 29)` |
| Conv2 | `(32, 64, 29, 29)` |
| Conv3 | `(32, 128, 29, 29)` |
| Conv4 | `(32, 256, 29, 29)` |
| MaxPool2 | `(32, 256, 28, 28)` |
| Dropout | `(32, 256, 28, 28)` |
| Flatten | `(32, 200704)` |
| FCN1 | `(32, 1000)` |
| FCN2 | `(32, 500)` |
| FCN3 + Softmax | `(32, 10)` |

**Note:** MaxPool1 gives 29 × 29 instead of 14 × 14 because the diagram uses stride 1 and padding 1. The code follows the diagram exactly.